# Dynamic pricing model deployement ready 


In [1]:
import pandas as pd 
import numpy as np
import sklearn
import pickle

In [2]:
df = pd.read_csv("dataset.csv")
df

,ID,SystemCodeNumber,Capacity,Latitude,Longitude,Occupancy,VehicleType,TrafficConditionNearby,QueueLength,IsSpecialDay,LastUpdatedDate,LastUpdatedTime
0,0,BHMBCCMKT01,577,26.144536,91.736172,61,car,low,1,0,04-10-2016,07:59:00
1,1,BHMBCCMKT01,577,26.144536,91.736172,64,car,low,1,0,04-10-2016,08:25:00
2,2,BHMBCCMKT01,577,26.144536,91.736172,80,car,low,2,0,04-10-2016,08:59:00
3,3,BHMBCCMKT01,577,26.144536,91.736172,107,car,low,2,0,04-10-2016,09:32:00
4,4,BHMBCCMKT01,577,26.144536,91.736172,150,bike,low,2,0,04-10-2016,09:59:00
...,...,...,...,...,...,...,...,...,...,...,...,...
18363,18363,Shopping,1920,26.150504,91.733531,1517,truck,average,6,0,19-12-2016,14:30:00
18364,18364,Shopping,1920,26.150504,91.733531,1487,car,low,3,0,19-12-2016,15:03:00
18365,18365,Shopping,1920,26.150504,91.733531,1432,cycle,low,3,0,19-12-2016,15:29:00
18366,18366,Shopping,1920,26.150504,91.733531,1321,car,low,2,0,19-12-2016,16:03:00


In [3]:
df.head()

,ID,SystemCodeNumber,Capacity,Latitude,Longitude,Occupancy,VehicleType,TrafficConditionNearby,QueueLength,IsSpecialDay,LastUpdatedDate,LastUpdatedTime
0,0,BHMBCCMKT01,577,26.144536,91.736172,61,car,low,1,0,04-10-2016,07:59:00
1,1,BHMBCCMKT01,577,26.144536,91.736172,64,car,low,1,0,04-10-2016,08:25:00
2,2,BHMBCCMKT01,577,26.144536,91.736172,80,car,low,2,0,04-10-2016,08:59:00
3,3,BHMBCCMKT01,577,26.144536,91.736172,107,car,low,2,0,04-10-2016,09:32:00
4,4,BHMBCCMKT01,577,26.144536,91.736172,150,bike,low,2,0,04-10-2016,09:59:00


In [4]:
df.describe()

,ID,Capacity,Latitude,Longitude,Occupancy,QueueLength,IsSpecialDay
count,18368.000000,18368.000000,18368.000000,18368.000000,18368.000000,18368.000000,18368.000000
mean,9183.500000,1605.214286,25.706547,90.751170,731.084059,4.587925,0.150915
std,5302.529208,1131.153886,1.582749,3.536636,621.164982,2.580062,0.357975
min,0.000000,387.000000,20.000035,78.000003,2.000000,0.000000,0.000000
25%,4591.750000,577.000000,26.140048,91.727995,322.000000,2.000000,0.000000
50%,9183.500000,1261.000000,26.147482,91.729511,568.000000,4.000000,0.000000
75%,13775.250000,2803.000000,26.147541,91.736172,976.000000,6.000000,0.000000
max,18367.000000,3883.000000,26.150504,91.740994,3499.000000,15.000000,1.000000


In [4]:
df.isnull().sum()

ID                        0
SystemCodeNumber          0
Capacity                  0
Latitude                  0
Longitude                 0
Occupancy                 0
VehicleType               0
TrafficConditionNearby    0
QueueLength               0
IsSpecialDay              0
LastUpdatedDate           0
LastUpdatedTime           0
dtype: int64

In [5]:
df['timestamp'] = pd.to_datetime(df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],  format='%d-%m-%Y %H:%M:%S')
df.drop(['LastUpdatedDate' ,'LastUpdatedTime' ],axis=1, inplace=True)
df['timestamp'].head()

0   2016-10-04 07:59:00
1   2016-10-04 08:25:00
2   2016-10-04 08:59:00
3   2016-10-04 09:32:00
4   2016-10-04 09:59:00
Name: timestamp, dtype: datetime64[ns]

In [7]:
df['Hour'] = df['timestamp'].dt.hour

In [8]:
df['DayofWeek'] = df['timestamp'].dt.dayofweek

In [9]:
df['Month'] = df['timestamp'].dt.month

In [10]:
df.head()

,ID,SystemCodeNumber,Capacity,Latitude,Longitude,Occupancy,VehicleType,TrafficConditionNearby,QueueLength,IsSpecialDay,timestamp,Hour,DayofWeek,Month
0,0,BHMBCCMKT01,577,26.144536,91.736172,61,car,low,1,0,2016-10-04 07:59:00,7,1,10
1,1,BHMBCCMKT01,577,26.144536,91.736172,64,car,low,1,0,2016-10-04 08:25:00,8,1,10
2,2,BHMBCCMKT01,577,26.144536,91.736172,80,car,low,2,0,2016-10-04 08:59:00,8,1,10
3,3,BHMBCCMKT01,577,26.144536,91.736172,107,car,low,2,0,2016-10-04 09:32:00,9,1,10
4,4,BHMBCCMKT01,577,26.144536,91.736172,150,bike,low,2,0,2016-10-04 09:59:00,9,1,10


In [11]:
df.drop(['timestamp'],axis=1 , inplace=True)
df.head()

,ID,SystemCodeNumber,Capacity,Latitude,Longitude,Occupancy,VehicleType,TrafficConditionNearby,QueueLength,IsSpecialDay,Hour,DayofWeek,Month
0,0,BHMBCCMKT01,577,26.144536,91.736172,61,car,low,1,0,7,1,10
1,1,BHMBCCMKT01,577,26.144536,91.736172,64,car,low,1,0,8,1,10
2,2,BHMBCCMKT01,577,26.144536,91.736172,80,car,low,2,0,8,1,10
3,3,BHMBCCMKT01,577,26.144536,91.736172,107,car,low,2,0,9,1,10
4,4,BHMBCCMKT01,577,26.144536,91.736172,150,bike,low,2,0,9,1,10


In [12]:
df.drop(['ID'], axis=1, inplace=True)
df.drop(['Latitude'], axis=1, inplace=True)
df.drop(['Longitude'], axis=1, inplace=True)
df.head()

,SystemCodeNumber,Capacity,Occupancy,VehicleType,TrafficConditionNearby,QueueLength,IsSpecialDay,Hour,DayofWeek,Month
0,BHMBCCMKT01,577,61,car,low,1,0,7,1,10
1,BHMBCCMKT01,577,64,car,low,1,0,8,1,10
2,BHMBCCMKT01,577,80,car,low,2,0,8,1,10
3,BHMBCCMKT01,577,107,car,low,2,0,9,1,10
4,BHMBCCMKT01,577,150,bike,low,2,0,9,1,10


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18368 entries, 0 to 18367
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   SystemCodeNumber        18368 non-null  object
 1   Capacity                18368 non-null  int64 
 2   Occupancy               18368 non-null  int64 
 3   VehicleType             18368 non-null  object
 4   TrafficConditionNearby  18368 non-null  object
 5   QueueLength             18368 non-null  int64 
 6   IsSpecialDay            18368 non-null  int64 
 7   Hour                    18368 non-null  int32 
 8   DayofWeek               18368 non-null  int32 
 9   Month                   18368 non-null  int32 
dtypes: int32(3), int64(4), object(3)
memory usage: 1.2+ MB


# to handle classes we are using one-hot encoding 

In [14]:
cat_cols = ['VehicleType', 'TrafficConditionNearby', 'SystemCodeNumber']

In [15]:
df = pd.get_dummies(df, columns = cat_cols)

In [16]:
df

,Capacity,Occupancy,QueueLength,IsSpecialDay,Hour,DayofWeek,Month,VehicleType_bike,VehicleType_car,VehicleType_cycle,...,SystemCodeNumber_BHMNCPHST01,SystemCodeNumber_BHMNCPNST01,SystemCodeNumber_Broad Street,SystemCodeNumber_Others-CCCPS105a,SystemCodeNumber_Others-CCCPS119a,SystemCodeNumber_Others-CCCPS135a,SystemCodeNumber_Others-CCCPS202,SystemCodeNumber_Others-CCCPS8,SystemCodeNumber_Others-CCCPS98,SystemCodeNumber_Shopping
0,577,61,1,0,7,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,577,64,1,0,8,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,False
2,577,80,2,0,8,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,False
3,577,107,2,0,9,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,577,150,2,0,9,1,10,True,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18363,1920,1517,6,0,14,0,12,False,False,False,...,False,False,False,False,False,False,False,False,False,True
18364,1920,1487,3,0,15,0,12,False,True,False,...,False,False,False,False,False,False,False,False,False,True
18365,1920,1432,3,0,15,0,12,False,False,True,...,False,False,False,False,False,False,False,False,False,True
18366,1920,1321,2,0,16,0,12,False,True,False,...,False,False,False,False,False,False,False,False,False,True


In [17]:
df['price'] = (10 + df['Occupancy']*0.05 + df['IsSpecialDay']*5 + df['QueueLength']*0.5)
df.head()

,Capacity,Occupancy,QueueLength,IsSpecialDay,Hour,DayofWeek,Month,VehicleType_bike,VehicleType_car,VehicleType_cycle,...,SystemCodeNumber_BHMNCPNST01,SystemCodeNumber_Broad Street,SystemCodeNumber_Others-CCCPS105a,SystemCodeNumber_Others-CCCPS119a,SystemCodeNumber_Others-CCCPS135a,SystemCodeNumber_Others-CCCPS202,SystemCodeNumber_Others-CCCPS8,SystemCodeNumber_Others-CCCPS98,SystemCodeNumber_Shopping,price
0,577,61,1,0,7,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,13.55
1,577,64,1,0,8,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,13.70
2,577,80,2,0,8,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,15.00
3,577,107,2,0,9,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,16.35
4,577,150,2,0,9,1,10,True,False,False,...,False,False,False,False,False,False,False,False,False,18.50


In [18]:
x = df.drop(['price'],axis=1)
y = df['price']

In [20]:
x = df.drop(['price'],axis=1)
y = df['price']

In [21]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split( x , y , test_size=0.2,random_state=42)

In [22]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor( n_estimators=100 , random_state=42)

In [23]:
model.fit(x_train,y_train)

RandomForestRegressor(random_state=42)

In [24]:
y_pred = model.predict(x_test)

In [25]:
from sklearn.metrics import r2_score, mean_absolute_error

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("R2 Score:", r2)
print("MAE:", mae)

R2 Score: 0.9999455600678544
MAE: 0.0918628198149172


In [18]:
important_df = pd.DataFrame({
    "Features":x.columns,
    "Importances" : model.feature_importances_
})

NameError: name 'x' is not defined

In [30]:
important_df = important_df.sort_values(
    by='Importances',
    ascending=False
)
print(important_df)

                             Features   Importances
1                           Occupancy  9.948655e-01
3                        IsSpecialDay  3.865370e-03
2                         QueueLength  1.139608e-03
12        TrafficConditionNearby_high  4.170538e-05
5                           DayofWeek  3.981256e-05
4                                Hour  1.449583e-05
13         TrafficConditionNearby_low  7.480905e-06
11     TrafficConditionNearby_average  6.681617e-06
6                               Month  4.162905e-06
0                            Capacity  3.111528e-06
8                     VehicleType_car  2.935047e-06
7                    VehicleType_bike  1.765222e-06
10                  VehicleType_truck  1.192934e-06
9                   VehicleType_cycle  1.142774e-06
21  SystemCodeNumber_Others-CCCPS105a  8.803699e-07
26    SystemCodeNumber_Others-CCCPS98  8.297702e-07
23  SystemCodeNumber_Others-CCCPS135a  7.112320e-07
24   SystemCodeNumber_Others-CCCPS202  4.153096e-07
22  SystemCo

In [30]:
x_new = df[['Occupancy','QueueLength','IsSpecialDay']]
y = df['price']

In [32]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_new , y , test_size=0.2 , random_state=42)

In [33]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(x_train,y_train)

RandomForestRegressor()

In [34]:
from sklearn.metrics import r2_score,mean_absolute_error
pred = model.predict(x_test)
print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

R2: 0.9999588838261961
MAE: 0.06699510070767754


In [36]:
import pickle 
pickle.dump(model, open("model.pkl","wb"))

In [37]:
loaded_model = pickle.load(open("model.pkl","rb"))

In [38]:
sample = x_test.iloc[[0]]
prediction = loaded_model.predict(sample)
print(prediction)

[28.748]


In [39]:
print("Prediction:",prediction)
print("Actual:",y_test.iloc[0])

Prediction: [28.748]
Actual: 28.75


In [31]:
print(x.columns)

Index(['Capacity', 'Occupancy', 'QueueLength', 'IsSpecialDay', 'Hour',
       'DayofWeek', 'Month', 'VehicleType_bike', 'VehicleType_car',
       'VehicleType_cycle', 'VehicleType_truck',
       'TrafficConditionNearby_average', 'TrafficConditionNearby_high',
       'TrafficConditionNearby_low', 'SystemCodeNumber_BHMBCCMKT01',
       'SystemCodeNumber_BHMBCCTHL01', 'SystemCodeNumber_BHMEURBRD01',
       'SystemCodeNumber_BHMMBMMBX01', 'SystemCodeNumber_BHMNCPHST01',
       'SystemCodeNumber_BHMNCPNST01', 'SystemCodeNumber_Broad Street',
       'SystemCodeNumber_Others-CCCPS105a',
       'SystemCodeNumber_Others-CCCPS119a',
       'SystemCodeNumber_Others-CCCPS135a', 'SystemCodeNumber_Others-CCCPS202',
       'SystemCodeNumber_Others-CCCPS8', 'SystemCodeNumber_Others-CCCPS98',
       'SystemCodeNumber_Shopping'],
      dtype='object')


In [29]:
print(model.n_features_in_)

3


In [32]:
x.iloc[[0]]

,Capacity,Occupancy,QueueLength,IsSpecialDay,Hour,DayofWeek,Month,VehicleType_bike,VehicleType_car,VehicleType_cycle,...,SystemCodeNumber_BHMNCPHST01,SystemCodeNumber_BHMNCPNST01,SystemCodeNumber_Broad Street,SystemCodeNumber_Others-CCCPS105a,SystemCodeNumber_Others-CCCPS119a,SystemCodeNumber_Others-CCCPS135a,SystemCodeNumber_Others-CCCPS202,SystemCodeNumber_Others-CCCPS8,SystemCodeNumber_Others-CCCPS98,SystemCodeNumber_Shopping
0,577,61,1,0,7,1,10,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [34]:
pd.set_option('display.max_columns', None)
print(x.iloc[[0]])

   Capacity  Occupancy  QueueLength  IsSpecialDay  Hour  DayofWeek  Month  \
0       577         61            1             0     7          1     10   

   VehicleType_bike  VehicleType_car  VehicleType_cycle  VehicleType_truck  \
0             False             True              False              False   

   TrafficConditionNearby_average  TrafficConditionNearby_high  \
0                           False                        False   

   TrafficConditionNearby_low  SystemCodeNumber_BHMBCCMKT01  \
0                        True                          True   

   SystemCodeNumber_BHMBCCTHL01  SystemCodeNumber_BHMEURBRD01  \
0                         False                         False   

   SystemCodeNumber_BHMMBMMBX01  SystemCodeNumber_BHMNCPHST01  \
0                         False                         False   

   SystemCodeNumber_BHMNCPNST01  SystemCodeNumber_Broad Street  \
0                         False                          False   

   SystemCodeNumber_Others-CCCPS1

In [35]:
x.shape

(18368, 28)

In [37]:
print(list(x.iloc[0].values))

[np.int64(577), np.int64(61), np.int64(1), np.int64(0), np.int32(7), np.int32(1), np.int32(10), np.False_, np.True_, np.False_, np.False_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_]
